# 03 報酬設計・割引率・探索率の落とし穴 — 強化学習は「報酬の言うとおり」にしか動かない

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 点数の付け方を誤ると、AIが予想外の抜け道を見つけることを体験する回です。
> - **たとえ話**: テストの点数だけ見て、中身を理解せず丸暗記する生徒。点数の決め方が行動を決める。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-6（報酬設計）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 15分 ＋ 実行 約2分半

## ゴール
- 報酬の設計を誤ると、エージェントが**予想外の抜け道**を見つける（**報酬ハッキング**）ことを体験する。
- **報酬がまばら**（ゴールだけ）だと学習が進まないこと、**楽観的な初期値**が助けになることを確かめる。
- **割引率γ**・**学習率α**・**探索率ε**が、結果にどう効くかを知る。

## 前のNotebookとのつながり
02 では、迷路・崖で、Q学習とSARSAが最適方策に近づくことを確かめました。報酬は「1歩ごとに −1、崖で −100」と、**うまく設計されていました**。
ここでは、報酬の**設計を変えたり崩したり**して、何が起きるかを調べます。

## キーワード
- **報酬ハッキング**: 人が意図したのと**違う方法**で、報酬を稼ぐ抜け道を、エージェントが見つけること。エージェントは、**報酬の最大化**だけをするので、人の意図を知りません。
- **まばらな報酬**: ゴールなど、ごく一部でしか報酬が出ないこと。途中の手がかりがない。
- **楽観的な初期値**: 価値の初期値を、実際より高くしておく（01 の腕と同じ）。まだ試していない行動が魅力的に見え、**自然に探索**が起きる。
- **割引率 γ**: 先の報酬を、どれだけ軽く見るか。γ=0.9 なら、1歩先の報酬は0.9倍、2歩先は0.81倍…。γ=1 は割引なし。
- **報酬整形（ポテンシャル型）**: ゴールへの近さ Φ に応じた**追加報酬**（γΦ(s') − Φ(s)）。**最適な方策を変えない**ことが、理論的に保証されている。
- **学習率 α**: 1回の更新で、目標にどれだけ近づけるか。大きすぎると不安定。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from rl_envs import make_env, value_iteration, evaluate_policy
from rl_envs.viz import draw_grid
import run_phase3

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (coin, goal) in zip(axes, [(0, 0), (1.5, 0), (5, 0)]):
    e = make_env("coin", coin_reward=coin, goal_reward=goal)
    V, pi = value_iteration(e, gamma=0.99)
    r = evaluate_policy(e, pi, 5)
    draw_grid(e, ax, policy=pi, title=f"コインの報酬 {coin}\n最適方策: ゴールに着く割合 {r['success_rate']:.0%}")
fig.suptitle("コインのある迷路の、最適方策（動的計画法・γ=0.99）", y=1.08)
fig.tight_layout()

## A. 報酬ハッキング — 「コインを何度でも拾える」迷路

迷路の真ん中に**コイン**（★）があります。目的は「ゴールに着くこと」ですが、**コインを踏むたびにボーナス**をあげる設計にします。
（ゴールまでの道は、コインのマスを通ります。）

**何をしているか**: コインの報酬を 0、1.5、5 にしたとき、**最適な方策（矢印）**を、動的計画法で求めて見ます。

**出力の見方**:
- コインの報酬が **0** や **1.5** のときは、ゴールに向かう最短ルートです（コインは、通りがかりに踏むだけ）。
- **5** のときは、矢印が、コインのマスと隣のマスの**間を往復**するようになります。ゴールに着く割合は **0%**。
  コインに乗って、隣へ出て、また戻れば、**2歩で（5−1）＋（−1）＝＋3**の得になります。ゴールするより、**往復を続けたほうが報酬が大きい**ので、エージェントは、**ゴールしません**。

In [ ]:
dfa = run_phase3.exp_a()
dfa

**何をしているか**: コインの報酬を変えながら、**最適方策**（動的計画法）と、**Q学習で学習した方策**を、比べます（10シード）。

**出力の見方**:
- コインの報酬が **1・1.5** までは、最適方策も、Q学習の結果も、**ゴールに着きます**（コインは1回だけ踏む）。
- **2.5以上**では、最適方策も、Q学習も、**ゴールに着きません**（ゴールに着く割合0.0）。コインを**99回**（200歩の半分）踏んでいます。
- 境目は、**コインの報酬 ＝ 2**です。往復の1周期（2歩）で得られる報酬は「コイン報酬 − 2」なので、**2を超えると**往復が得になります。
- **Q学習は、最適方策と同じ抜け道を、自分で見つけます**（平均リターンが、コインの報酬が大きいほど大きい）。**学習は、報酬の最大化を、忠実にやっている**のです。
- 一番下の行（コイン5＋ゴールの報酬500）は、**ゴールの報酬を大きく**すると、ゴールを選ぶように戻る例です。報酬の**バランスが大事**です。
- 教訓: **報酬は、「人の意図」ではなく「測れるもの」で決まる**。**設計したら、方策を目で見て、意図どおりか確かめる**。

In [ ]:
dfb = run_phase3.exp_b()
dfb

## B. 報酬がまばらなとき

**長い一本道**（snake。最短53歩）で、**ゴールにだけ報酬**がある場合を調べます。途中には、何の手がかりもありません。

**何をしているか**: 6つの設計で、Q学習を1500エピソード学習させます（10シード）。「ゴールできたシード数」「最初にゴールしたエピソード」「学習後の方策がゴールに着く割合」を見ます。

**出力の見方**:
- **まばら・Q初期値0**: 1500エピソードのうち、**ゴールを見つけられたのは10シード中1つだけ**。方策は、ゴールに着きません。偶然ゴールに着くのを、待つしかないのです（53歩の一本道を、ランダムに歩いて着くのは、とても難しい）。
- **まばら・Q初期値1（楽観的）**: **10シードすべて、約11エピソードでゴール**し、方策も**最適（53歩）**。初期値が高いと、**試した行動は、報酬が0なので、価値が下がり**、まだ試していない行動のほうが魅力的に見えます。結果として、**系統的に探索**して、ゴールを見つけます（01 の楽観的初期値と同じ）。
- **まばら・Q初期値1・割引なし（γ=1）**: 初期値が楽観的でも、**割引がないと効きません**（1シードのみ）。割引なしでは、ゴールの価値が1のまま減らず、初期値1が、**もう楽観的でなくなる**ためです。
- **1歩ごとに−1・Q初期値0**: 10シードとも約11.5エピソードでゴール。**1歩ごとの罰（−1）がある設計は、Q初期値0が自然と楽観的**になるため、うまくいきます。
- **1歩ごとに−1・Q初期値−100（悲観的）**: 初期値を悲観的にすると、**また失敗**します（1シードのみ）。つまり、**「罰が毎歩ある」ことよりも、「初期値が楽観的」なこと**が効いていました。
- **まばら＋整形（ポテンシャル型）**: 整形しても、**1シードも、ゴールできません**。次で理由を見ます。

In [ ]:
base = make_env("snake", step_reward=0.0, goal_reward=1.0, max_steps=300)
shaped = run_phase3.make_b_env("shaped", 0)
s = base.start
print("スタートで『右』（ゴールに近づく）の報酬:", [round(o[2], 2) for o in shaped.P[s][1]])
print("スタートで『左』（壁にぶつかってその場にとどまる）の報酬:", [round(o[2], 2) for o in shaped.P[s][3]])

**何をしているか**: 整形した報酬が、「ゴールに近づく」と「その場にとどまる」で、どうなっているかを見ます。

**出力の見方**: 近づくと +1.52、**壁にぶつかってその場にとどまっても +0.53** です。割引率γ=0.99 の整形では、**とどまるだけで、ごくわずかに報酬**が出てしまいます（γΦ(s) − Φ(s) = 0.01 × 距離 > 0）。
理論上、整形しても**最適な方策**は変わりません（ゴールに着く方策が、最もリターンが大きい）。しかし、**ゴールに着く方策と、その場にとどまる方策の価値の差が、とても小さい**（約1%）ので、ゴールを知らない学習初期には、**区別がつきません**。
また、整形は、**各状態の全ての行動に、同じ値（−Φ(s)）を足す**だけなので、**行動の順位は変わりません**。この例では、**整形は、助けになりませんでした**。
**「理論上、安全」でも、「学習が速くなる」とは限らない**、という教訓です。まばらな報酬には、**楽観的な初期値**のほうが、はるかに効きました。

In [ ]:
dfc = run_phase3.exp_c()
dfc

## C. 割引率γ

**すべる崖（0.2）**で、割引率γを変えて、動的計画法で最適方策を求めます。そして、その方策の**実際のリターン**（割引なし）を測ります。

**出力の見方**:
- **最適な価値V\*（スタート）**は、γで**大きく変わります**（γ=1 で −29.77、γ=0.5 で −2.00、γ=0.1 で −1.11）。γが小さいほど、**先の報酬を軽く見る**ので、価値の見積もりは、0に近くなります。
- ところが、**方策は、γ=0.8以上では、同じ**です（実際のリターンは −30.65 で変わらない）。**割引率は、価値の「大きさ」を変えますが、十分大きければ最適な方策は変えません**。
- **γが小さすぎる**（0.1、0.5）と、**近視眼的**になり、実際のリターンが悪化します（−54.69、−31.85）。先の費用（崖に落ちる罰など）を軽く見て、**危険な近道**を選ぶためです。
- 割引率は、**「どれだけ先を見て決めるか」**を決めます。目的に合わせて選ぶ（長い目で見るなら、1に近く）。

In [ ]:
dfd = run_phase3.exp_d()
piv = dfd.pivot(index="学習率α", columns="探索率ε", values="学習後の成績(平均)")
fig, ax = plt.subplots(figsize=(6.4, 4))
im = ax.imshow(np.clip(piv.values, -80, -29), cmap="viridis", aspect="auto")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns); ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
ax.set(xlabel="探索率ε", ylabel="学習率α", title="すべる崖(0.2)・Q学習・1000エピソード\n学習後の方策の成績（最適 −29.8、色は−80でカット）")
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f"{piv.values[i, j]:.0f}", ha="center", va="center", color="w", fontsize=9)
fig.colorbar(im); fig.tight_layout()
dfd.pivot(index="学習率α", columns="探索率ε", values="ばらつき(sd)").rename_axis("学習率α（ばらつきsd）")

## D. 学習率αと探索率ε

**すべる崖（0.2）**で、学習率αと探索率εの**組み合わせ**を変えて、Q学習の**学習後の方策の成績**を見ます（10シード、1000エピソード。最適は −29.8）。数字は平均、下の表は**シード間のばらつき（標準偏差）**です。

**出力の見方**:
- **学習率αが大きい（0.5〜1.0）**と、成績が**悪く、ばらつきも大きい**（α=1.0 で −63〜−369、sdが数十〜数百）。すべる環境では、**1回の結果（たまたま滑った）に引きずられて、更新がぶれる**ためです。
- **αが小さい（0.05〜0.3）**と、**比較的安定して最適に近く**（−32〜−36。sdは多くが2前後で、最大でも5〜6）。02 で、500エピソード・α=0.5 では最適（−29.8）に届かなかった理由が、これです。
- **探索率εは、αほど効きません**（α=0.1 では、ε=0.01〜0.3 で −31.6〜−32.4）。ただし、**αが大きいときにεも大きい**（α=0.5・ε=0.3）と、**さらに悪化**（−64.8、sd 47）します。
- **確率的な環境では、学習率を小さく**する。代わりに、**エピソードが必要**（学習が遅い）。
- どの組み合わせも、最適（−29.8）に**完全には届かない**（最良でも −31.5）。**1000エピソードでは、まだ足りない**か、**表形式の限界**（学習率が一定のため、ゆらぎが残る）です。

## 試してみよう（演習）
1. **ハッキングの境目を見つける**: `run_phase3.exp_a` の、コインの報酬のリスト `[(0, 0), (1, 0), (1.5, 0), (2.5, 0), ...]` に `(2, 0)` と `(2.2, 0)` を足してください。境目（2）の前後で、方策が変わることを確かめます。
2. **ゴールの報酬で直す**: `(5, 500)` を `(5, 50)` と `(5, 200)` にして、ゴールを選ぶ最小のゴール報酬を探してください。分かるのは、ハッキングを防ぐには、**ゴールの報酬を十分大きく**する必要があること（ここでは、往復の稼ぎの割引総和を上回る必要）です。
3. **楽観的な初期値を変える**: `B_SPECS` の `q0=1.0` を `0.3`、`3.0` にして、ゴールまでのエピソード数を比べてください。分かるのは、初期値が高いほど探索が長く続くこと（高すぎると、無駄が増える）です。
4. **割引率を変えて、まばらな報酬を試す**: `B_SPECS` の `0.99` を `0.9` にしてください。分かるのは、γが小さいと、ゴールまでの価値が小さくなり、**楽観的初期値の効果が変わる**ことです。
5. **学習率を下げ、エピソードを増やす**: `exp_d` の `episodes=1000` を `5000` にして、α=0.05 と0.1 で、最適（−29.8）に近づくか確かめてください。

## まとめ
- **報酬ハッキング**: エージェントは、**報酬を最大化**するだけで、人の意図は知らない。コインの報酬が **2** を超えると、ゴールせずに往復する（Q学習も、自分で見つける）。**設計したら、方策を目で見て確かめる**。
- **報酬がまばら**だと、学習が進まない（1500エピソードで、10シード中1つ）。**楽観的な初期値**で、全シードが約11エピソードでゴール。ただし、**割引がないと効かない**。「1歩ごとの罰」が効く理由も、実は**Q=0が楽観的**だから（悲観的にすると失敗）。
- **ポテンシャル型の整形**は、最適方策を変えないが、**学習が速くなるとは限らない**（ここでは助けにならなかった）。
- **割引率**は、価値の大きさを変えるが、**十分大きければ方策は変えない**。**小さすぎる（0.1）と近視眼的**で、実際のリターンが悪化（−30.7 → −54.7）。
- **学習率**は、確率的な環境では**小さく**（0.05〜0.3）。大きい（1.0）と、不安定。探索率は、学習率ほど効かない。

次の Phase4 では、状態が**連続**（座標が小数）で、表に収まらない問題を、**ニューラルネットワーク**（DQN）で解きます。